# Q1(d) — fitting, simulating, and judging adequacy  ·  5 marks

*Scratchpad, not submission material. Invented numbers throughout.*

> Fit the preferred model from part c), and use posterior simulation to estimate the posterior
> predictive $p$-value based on $T$. Briefly comment on whether the most probable model also
> appears to be an adequate model.

| # | Deliverable | Note |
|---|---|---|
| 1 | Fit + simulate + estimate $\hat p$ | the posterior is *given to you* in the question |
| 2 | Comment on adequacy | this is where the marks hide |

Deliverable 1 is largely mechanical: the question hands you $V_n, m_n, a_n, b_n$ directly, so
no derivation is needed. Which means the discriminating part is the comment.

## The bug that catches everyone: NumPy's Gamma parameterisation

You need $\sigma^{-2}\sim\Gam(a_n,b_n)$ where $b_n$ is a **rate**. NumPy's `gamma` takes a
**scale**. Get this wrong and everything downstream is silently plausible but incorrect.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
a_n, b_n = 7.0, 8.0          # invented
N = 200_000

wrong = rng.gamma(shape=a_n, scale=b_n, size=N)      # treats b_n as a scale
right = rng.gamma(shape=a_n, scale=1 / b_n, size=N)  # b_n is a rate

print(f"analytic  E[sigma^-2] = a_n / b_n = {a_n / b_n:.4f}")
print(f"scale=b_n   -> sample mean {wrong.mean():8.4f}   <- off by a factor of b_n^2")
print(f"scale=1/b_n -> sample mean {right.mean():8.4f}   <- matches")

## Verifying your sampler before you trust $\hat p$

The posterior has known moments, so check the sampler against them. If these do not match,
the $p$-value is meaningless no matter how sensible it looks.

In [ ]:
prec = right                       # sigma^{-2} draws
sigma2 = 1 / prec

print(f"E[sigma^-2] : analytic {a_n / b_n:8.4f}   sample {prec.mean():8.4f}")
print(f"Var[sigma^-2]: analytic {a_n / b_n**2:8.4f}   sample {prec.var():8.4f}")
print(f"E[sigma^2]  : analytic {b_n / (a_n - 1):8.4f}   sample {sigma2.mean():8.4f}")
print("\n(the last one needs a_n > 1 -- why?)")

**Q1.** For $\beta$, what is the analogous check? The question gives you $\E[\beta\mid\sigma^2,\y]$
directly — compare it against your sample mean of the $\beta$ draws.

**Q2.** Why must $\beta$ be drawn *conditionally* on each $\sigma^2$ draw, rather than once at
a fixed $\sigma^2$? What would the $\beta$ marginal look like if you did it correctly?

## Reading $\hat p$ — the part worth marks

Recall from (b) that misfit shows at both ends. Here is the asymmetry that makes the comment
interesting: compare where the *observed* discrepancy sits relative to the replicated ones.

In [ ]:
def summarise(T_obs, T_rep, label):
    p = np.mean(T_rep >= T_obs)
    se = np.sqrt(p * (1 - p) / len(T_obs))
    print(f"{label:34s} p-hat = {p:.4f} (se {se:.4f})   "
          f"E[T_obs] = {T_obs.mean():6.2f}   E[T_rep] = {T_rep.mean():6.2f}")


M = 20_000
for label, scale in (("well calibrated", 1.0),
                     ("p-hat high", 0.32),
                     ("p-hat low", 2.6)):
    T_rep = rng.chisquare(df=9, size=M)
    summarise(scale * rng.chisquare(df=9, size=M), T_rep, label)

**Q3.** In the "p-hat high" row, $\E[T_{\text{obs}}]$ sits well below $\E[T_{\text{rep}}]$.
State what that means about the model's predicted residual scatter versus the actual scatter.
Give it a name.

**Q4.** If you find $\hat p$ high, the natural follow-up is *why*. With $n=10$ and
$\lambda=1, a=b=2$, is the prior doing much work here? Two things to look at:

- compare $\E[\sigma^2\mid\y]$ with the realised mean squared residual $\E[T(\y,\beta)\mid\y]/n$;
- compare the posterior mean of the intercept with its least-squares value.

Both are one line each once you have the draws, and either would support a sentence in the
comment. This also sets up part (e).

**Q5.** "Most probable" and "adequate" are different claims. Make sure your comment answers
the question actually asked — is the winner of part (c) *adequate*, not merely the winner.

---
## Your turn

Implement between `# <<q1d` / `# >>q1d` in `code/q1_model_choice.py`, export via
`write_results`, and build the two-panel figure (posterior fit; distribution of
$T(\yrep,\beta)$ against $T(\y,\beta)$) with `savefig(fig, "q1_ppc")` — the report already has
the `\includegraphics` block commented out and ready.

The adequacy comment (aim: 3–4 sentences — state $\hat p$, say whether it indicates misfit,
and if it is off-centre say in which direction and what you think is causing it):

*(your text here)*